In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# question 1. Justifying the Alignment Unit


My Choice: Word-level alignment.

The Justification: Subwords (like characters) are too fragmented. For example, "14" and "चौदह" share absolutely zero characters, so subword alignment would fail to see they are the same concept. Phrase-level is too rigid because humans don't swap whole phrases; they swap individual words or numbers. The Word-level is the perfect semantic unit because variations (synonyms, number formatting, spelling variants) naturally occur one word at a time.

# Question 2. When to trust models over the reference

Human annotators make mistakes, or they enforce rigid formatting (like always using digits instead of words). We need a rule to override the reference.

The Approach: Use a Voting Threshold. If a model outputs a word that doesn't match the reference, we count how many models agree on that specific variation. If 2 or more models (out of 5) agree on an alternative spelling or format, we assume it is a valid variant and add it to the Lattice Bin. If only 1 model says it, we assume it's a hallucination/error and do not add it.

# Question 3. How to handle Insertions, Deletions, and Substitutions
When building the lattice, we align every model to the human reference.

Substitutions: If the reference is "तेज़", but 2 models say "तेज", we add "तेज" to that bin. The bin becomes ["तेज़", "तेज"].

Deletions (by the model): If models skip a word entirely (e.g., in audio_002, Model 4 drops the "ना" at the end), we add a blank <eps> (epsilon) token to the bin if enough models agree that the word is unnecessary.

Insertions (by the model): If models add a word not in the reference (like "पाँच सौ" instead of "500"), we expand the lattice to have parallel paths.

In [5]:
import pandas as pd
import numpy as np
import difflib

# Load the data
df = pd.read_excel("/content/Question 4.xlsx")

def build_advanced_lattice(reference, models):
    ref_words = reference.split()

    # 1. Initialize trackers for the Democracy Rule
    # Track variants and deletions for existing reference words
    lattice_bins = [{'ref': w, 'variants': {w: 5}, 'deletions': 0} for w in ref_words]

    # Track newly inserted words (index 'i' means inserted BEFORE ref_words[i])
    insertions = {i: {} for i in range(len(ref_words) + 1)}

    # 2. Align each model against the reference to gather votes
    for model_text in models:
        hyp_words = str(model_text).split()
        sm = difflib.SequenceMatcher(None, ref_words, hyp_words)

        for tag, i1, i2, j1, j2 in sm.get_opcodes():
            if tag == 'replace' or tag == 'equal':
                # Model substituted a word (or matched)
                for offset in range(max(i2-i1, j2-j1)):
                    r_idx, h_idx = i1 + offset, j1 + offset
                    if r_idx < i2 and h_idx < j2:
                        h_word = hyp_words[h_idx]
                        lattice_bins[r_idx]['variants'][h_word] = lattice_bins[r_idx]['variants'].get(h_word, 0) + 1
                    elif r_idx < i2: # Model deleted a reference word here
                        lattice_bins[r_idx]['deletions'] += 1
                    elif h_idx < j2: # Model inserted an extra word here
                        h_word = hyp_words[h_idx]
                        insertions[i2][h_word] = insertions[i2].get(h_word, 0) + 1

            elif tag == 'delete':
                # Model skipped a reference word completely
                for offset in range(i2 - i1):
                    lattice_bins[i1 + offset]['deletions'] += 1

            elif tag == 'insert':
                # Model added a brand new word
                for offset in range(j2 - j1):
                    h_word = hyp_words[j1 + offset]
                    insertions[i1][h_word] = insertions[i1].get(h_word, 0) + 1

    # 3. Build the final Lattice using the Voting Threshold (>= 2 models)
    final_lattice = []
    for i in range(len(ref_words) + 1):
        # A. Create bins for VALIDATED INSERTIONS
        valid_inserts = [w for w, count in insertions[i].items() if count >= 2]
        if valid_inserts:
            # We add '<eps>' (blank) so models that match the old reference aren't penalized
            final_lattice.append({
                'type': 'insertion',
                'valid_words': set(valid_inserts).union({'<eps>'})
            })

        # B. Finalize bins for REFERENCE WORDS
        if i < len(ref_words):
            valid_alts = [w for w, count in lattice_bins[i]['variants'].items() if count >= 2]
            valid_set = set(valid_alts)

            # If >= 2 models deleted this word, the reference is likely wrong. Allow '<eps>' (skip).
            if lattice_bins[i]['deletions'] >= 2:
                valid_set.add('<eps>')

            final_lattice.append({
                'type': 'reference',
                'ref_word': lattice_bins[i]['ref'],
                'valid_words': valid_set
            })

    return final_lattice

# 4. Calculate Lattice WER using Dynamic Programming
def calculate_lattice_wer(lattice, hypothesis):
    hyp_words = hypothesis.split()
    n_bins = len(lattice)
    n_hyp = len(hyp_words)

    # Standard WER Matrix
    dp = np.zeros((n_bins + 1, n_hyp + 1))

    for i in range(1, n_bins + 1):
        # If the lattice bin allows <eps>, we can move past it with ZERO penalty!
        if '<eps>' in lattice[i-1]['valid_words']:
            dp[i][0] = dp[i-1][0]
        else:
            dp[i][0] = dp[i-1][0] + 1

    for j in range(1, n_hyp + 1):
        dp[0][j] = j

    for i in range(1, n_bins + 1):
        for j in range(1, n_hyp + 1):
            valid_words = lattice[i-1]['valid_words']
            hyp_w = hyp_words[j-1]

            # 1. Match/Substitution
            match = dp[i-1][j-1] if hyp_w in valid_words else dp[i-1][j-1] + 1

            # 2. Insertion
            insertion = dp[i][j-1] + 1

            # 3. Deletion (If '<eps>' is in valid words, the model is allowed to skip it for free!)
            deletion = dp[i-1][j] if '<eps>' in valid_words else dp[i-1][j] + 1

            dp[i][j] = min(match, insertion, deletion)

    # Normalize by the number of original reference words
    ref_length = sum(1 for b in lattice if b['type'] == 'reference')
    return dp[n_bins][n_hyp] / ref_length if ref_length > 0 else 0

# --- EXECUTE ON YOUR DATA ---
print("Running Advanced Lattice Evaluation...\n")

for index, row in df.iterrows():
    audio_id = row['segment_url_link']
    reference = row['Human']
    models = [row['Model H'], row['Model i'], row['Model k'], row['Model l'], row['Model m'], row['Model n']]

    print(f"--- {audio_id} ---")
    lattice = build_advanced_lattice(reference, models)

    print("Generated Lattice Bins:")
    for i, bin_data in enumerate(lattice):
        print(f"  Bin {i}: {list(bin_data['valid_words'])}")

    print("\nResults:")
    for m_idx, model_text in enumerate(models):
        lattice_wer = calculate_lattice_wer(lattice, str(model_text))
        print(f"  Model {m_idx+1}: Lattice WER = {lattice_wer:.2f}")
    print("\n")

Running Advanced Lattice Evaluation...

--- https://storage.googleapis.com/testing_audio_for_josh/bcmk_TR/hindi/ch_1726922_866.16_868.20.wav ---
Generated Lattice Bins:
  Bin 0: ['वही']
  Bin 1: ['अपना']
  Bin 2: ['खेती']
  Bin 3: ['बाड़ी']
  Bin 4: ['और']
  Bin 5: ['क्या']

Results:
  Model 1: Lattice WER = 0.00
  Model 2: Lattice WER = 0.17
  Model 3: Lattice WER = 0.17
  Model 4: Lattice WER = 0.00
  Model 5: Lattice WER = 0.33
  Model 6: Lattice WER = 0.00


--- https://storage.googleapis.com/testing_audio_for_josh/bcmk_TR/hindi/ch_2052946_52.62_54.36.wav ---
Generated Lattice Bins:
  Bin 0: ['मौनता']
  Bin 1: ['का']
  Bin 2: ['हर', 'अर्थ']
  Bin 3: ['क्या']
  Bin 4: ['होता']
  Bin 5: ['<eps>', 'है']

Results:
  Model 1: Lattice WER = 0.00
  Model 2: Lattice WER = 0.17
  Model 3: Lattice WER = 0.83
  Model 4: Lattice WER = 0.33
  Model 5: Lattice WER = 0.67
  Model 6: Lattice WER = 0.17


--- https://storage.googleapis.com/testing_audio_for_josh/bcmk_TR/hindi/ch_2054042_186.66_189.

It Builds the Lattice: For audio, the reference is "तेज़". Because Models 1 and 5 say "तेज", the frequency is $\ge 2$. The code adds both to the bin: ['तेज़', 'तेज'].


It recalculates WER: Now, when Model 1 is evaluated against the lattice, the algorithm sees "तेज" in the bin and scores a 0 (Match) instead of a 1 (Substitution error).


It reduces WER for unfairly penalized models: As requested by the prompt, models that were heavily penalized for minor spelling variations or number expansions will see their Lattice WER drop significantly compared to their Standard WER, while genuinely bad models will remain unchanged!